# Interpretación de $K$ y $\Delta t$

Este notebook reúne los perfiles fluviales, las terrazas revisadas
y los knickpoints seleccionados para definir los intervalos que
utilizaremos en la interpretación.

El trabajo se organiza en tres etapas:

1. **Definir los intervalos:** registrar qué terraza, límite inferior
   y knickpoint se utilizarán en cada río.
2. **Evaluar los perfiles:** comparar los ajustes en $\chi$–elevación
   para distintas concavidades $\theta$, manteniendo fijos los límites.
3. **Plantear escenarios:** estimar $K$ y, cuando corresponda,
   un tiempo de propagación $\Delta t$, documentando los supuestos.

Distinguimos el tramo costero que podría utilizarse para calibrar $K$
del tramo superior que llega hasta el knickpoint. Aplicar un valor de
$K$ fuera del intervalo donde se calibró constituye una extrapolación.

Las terrazas seleccionadas tienen edades asignadas por correlación.
Conservaremos su respaldo y las discrepancias cronológicas sin
convertirlas automáticamente en dataciones directas del punto.

La selección de un intervalo y un buen ajuste geométrico no demuestran,
por sí solos, que el río se encuentre en equilibrio.

## 1. Cargar los resultados anteriores

Cargamos los perfiles exportados, el catálogo de knickpoints y las
planillas de terrazas. Este notebook utiliza esos archivos como
entradas y no modifica sus datos originales.

El catálogo de knickpoints debe corresponder a la exportación que
incluye nuestras selecciones manuales. El caché de selección y el
catálogo exportado son archivos diferentes: guardar un punto en el
caché no actualiza una exportación anterior.

Los IDs de los ríos de la planilla de terrazas no coinciden
necesariamente con los IDs de los perfiles. Por ello, relacionaremos
ambos archivos por el nombre del cauce y revisaremos la correspondencia.

Todas las distancias y elevaciones se trabajan en metros, y las
áreas aportantes en metros cuadrados.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid
from scipy.stats import linregress
from IPython.display import display

# ── RUTAS ────────────────────────────────────────────────────────
MEMORIA = Path(
    r"C:\datos\Memoria"
)
PRODUCTOS = MEMORIA / "Codigos Stanford" / "Productos"

CARPETA_PERFILES = PRODUCTOS / "info_perfiles" / "20260924"

ARCHIVO_TERRAZAS = MEMORIA / "Terrazas_revisadas_UTM18S.xlsx"

ARCHIVO_REVISION = Path(
    r"C:\datos\capas_externas\Revision_muestras_Jara_2015.xlsx"
)

# MODIFICAR AQUÍ:
# Escribe la carpeta de exportación que contiene tus knickpoints manuales.
# Si dejas None, se usará la exportación más reciente y se mostrará su ruta.
CARPETA_KP = None
# CARPETA_KP = PRODUCTOS / "analisis_knickpoints" / "nombre_de_tu_carpeta"

if CARPETA_KP is None:
    archivos = sorted(
        (PRODUCTOS / "analisis_knickpoints").glob(
            "*/catalogo_candidatos.csv"
        )
    )

    if not archivos:
        raise FileNotFoundError(
            "No hay catálogos exportados. Ejecuta primero la exportación "
            "de AnalisisPerfiles después de seleccionar los manuales."
        )

    ARCHIVO_KP = archivos[-1]
else:
    ARCHIVO_KP = Path(CARPETA_KP) / "catalogo_candidatos.csv"

ARCHIVO_MANIFEST = CARPETA_PERFILES / "manifest.json"

for archivo in [
    ARCHIVO_MANIFEST,
    ARCHIVO_KP,
    ARCHIVO_TERRAZAS,
    ARCHIVO_REVISION,
]:
    if not archivo.is_file():
        raise FileNotFoundError(f"No se encontró:\n{archivo}")

print("Catálogo que se utilizará:")
print(ARCHIVO_KP)

In [ ]:
manifest = json.loads(
    ARCHIVO_MANIFEST.read_text(encoding="utf-8")
)

if (
    manifest["unidad_distancia"] != "m"
    or manifest["unidad_area"] != "m2"
):
    raise ValueError("Se requieren distancias en m y áreas en m².")

perfiles = {}
nombres = {}

columnas_perfil = [
    "distancia_m", "elevacion_m", "area_m2", "x_utm", "y_utm"
]

for rid, datos in manifest["rios"].items():
    rid = str(rid)
    p = pd.read_csv(CARPETA_PERFILES / datos["archivo"])

    faltantes = set(columnas_perfil) - set(p.columns)

    if faltantes:
        raise ValueError(f"Río {rid}: faltan columnas {faltantes}.")

    if not np.isfinite(p[columnas_perfil].to_numpy(float)).all():
        raise ValueError(
            f"Río {rid}: hay datos no finitos. Revisar antes de continuar."
        )

    if (
        len(p) < 3
        or np.any(np.diff(p["distancia_m"]) <= 0)
        or np.any(p["area_m2"] <= 0)
    ):
        raise ValueError(f"Río {rid}: revisar distancias y áreas.")

    perfiles[rid] = p
    nombres[rid] = datos["nombre"]

knickpoints = pd.read_csv(ARCHIVO_KP, dtype={"ID": str})

requeridas_kp = {
    "ID", "id_kp", "indice", "metodo",
    "distancia_m", "elevacion_m", "x_utm", "y_utm",
}

if requeridas_kp - set(knickpoints.columns):
    raise ValueError("El catálogo no tiene todas las columnas requeridas.")

if knickpoints["id_kp"].duplicated().any():
    raise ValueError("El catálogo contiene IDs de knickpoints duplicados.")

# Verificar que cada candidato corresponde al perfil cargado.
for _, k in knickpoints.iterrows():
    rid = k["ID"]

    if rid not in perfiles:
        raise ValueError(f"El catálogo incluye un río no cargado: {rid}")

    indice = float(k["indice"])

    if not np.isfinite(indice) or not indice.is_integer():
        raise ValueError(f"Índice inválido: {k['id_kp']}")

    j = int(indice)
    p = perfiles[rid]

    if not 0 <= j < len(p):
        raise ValueError(f"Índice fuera del perfil: {k['id_kp']}")

    columnas_control = [
        "distancia_m", "elevacion_m", "x_utm", "y_utm"
    ]

    if not np.allclose(
        p.iloc[j][columnas_control].to_numpy(float),
        k[columnas_control].to_numpy(float),
        atol=0.1,
        rtol=0,
    ):
        raise ValueError(
            f"{k['id_kp']}: el catálogo y el perfil no coinciden."
        )

terrazas_originales = pd.read_excel(
    ARCHIVO_TERRAZAS, sheet_name="Hoja1"
)
revision_muestras = pd.read_excel(
    ARCHIVO_REVISION, sheet_name="Resumen"
)

n_manuales = int(knickpoints["metodo"].eq("Manual").sum())

print(f"Perfiles cargados: {len(perfiles)}")
print(f"Knickpoints del catálogo: {len(knickpoints)}")
print(f"Selecciones manuales incluidas: {n_manuales}")

if n_manuales == 0:
    print(
        "Este catálogo no contiene puntos manuales. "
        "Si esperabas encontrarlos, revisa la carpeta de exportación."
    )

## 2. Relacionar las terrazas con los perfiles

Utilizamos las asociaciones río–terraza registradas en la planilla
revisada. No buscamos automáticamente la terraza más cercana ni
escogemos la de mayor elevación.

La misma terraza puede estar propuesta para más de un río. Por eso,
cada asociación se identifica mediante el ID del perfil y el ID
de la terraza.

Conservamos por separado:

- La elevación y su incertidumbre.
- La identificación del marcador.
- Las muestras citadas como respaldo.
- El carácter local o correlacionado de la edad.
- Las observaciones sobre la asociación.

Los registros sin coordenadas, sin elevación o sin un perfil
correspondiente se mantienen en una tabla aparte para revisión.
No se les asignan valores sustitutivos.

In [ ]:
def normalizar_nombre(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(
        c for c in texto if not unicodedata.combining(c)
    )
    texto = texto.lower().strip()
    texto = re.sub(r"^s/n\s*", "", texto)
    texto = re.sub(r"^(rio|estero)\s+", "", texto)
    texto = re.sub(r"\s+", " ", texto)

    alias = {
        "buchupero": "buchupureo",
        "cobequecura": "cobquecura",
    }
    return alias.get(texto, texto)


nombre_a_id = {
    normalizar_nombre(nombre): rid
    for rid, nombre in nombres.items()
}

if len(nombre_a_id) != len(nombres):
    raise ValueError("Hay nombres de río ambiguos.")

t = terrazas_originales.copy()
t["fila_excel"] = np.arange(len(t)) + 2
t["Cauce"] = t["Cauce"].ffill()

t["cauce_excel"] = t["Cauce asociado"].fillna(t["Cauce"])
t["ID"] = t["cauce_excel"].map(
    lambda nombre: nombre_a_id.get(normalizar_nombre(nombre))
)

t["terraza_id"] = (
    t["Terraza"].astype("string")
    .str.strip()
    .str.replace("-", "_", regex=False)
)

columnas_numericas = [
    "Elevación (m)", "Error elevación (m)",
    "Este UTM18S (m)", "Norte UTM18S (m)",
]

for columna in columnas_numericas:
    t[columna] = pd.to_numeric(t[columna], errors="coerce")

datos_validos = np.isfinite(t[[
    "Elevación (m)", "Este UTM18S (m)", "Norte UTM18S (m)"
]].to_numpy(float)).all(axis=1)

asociacion_valida = (
    datos_validos
    & t["ID"].notna()
    & t["terraza_id"].notna()
)

terrazas = t.loc[asociacion_valida].copy()
terrazas_no_asociadas = t.loc[~asociacion_valida].copy()

terrazas["asociacion_id"] = (
    "R" + terrazas["ID"].str.zfill(2)
    + "_" + terrazas["terraza_id"]
)

if terrazas["asociacion_id"].duplicated().any():
    raise ValueError("Hay asociaciones río–terraza duplicadas.")

if not terrazas["CRS coordenadas UTM"].astype(str).str.contains(
    "32718", regex=False
).all():
    raise ValueError("Revisa el CRS de las terrazas.")

#display(
 #   terrazas[[
  #      "ID", "cauce_excel", "terraza_id",
   #     "Elevación (m)", "Error elevación (m)",
    #    "Edad local o correlacionada",
    #]]
#)

print(f"Registros apartados para revisión: {len(terrazas_no_asociadas)}")

## 3. Visualizar todas las terrazas y los posibles knickpoints

Mostramos los perfiles longitudinales y χ–elevación con todas las
terrazas asociadas a cada río y todos los candidatos del catálogo cargado.

Las líneas horizontales representan la elevación de las terrazas;
las bandas muestran su incertidumbre. No representan su extensión
espacial ni un límite de análisis seleccionado.

Esta visualización permite revisar las alternativas antes de registrar
la selección manual en Excel.

In [ ]:
# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
RIOS_VER = list(perfiles)
# RIOS_VER = ["6"]

THETA_VISUAL = 0.45
A0_VISUAL_M2 = 1_000_000.0
MOSTRAR_TABLAS = True


def graficar_terrazas_kp(
    rid, terrazas_rio, kp_rio,
    titulo="Alternativas disponibles",
    seleccion=False,
):
    p = perfiles[rid]

    x = p["distancia_m"].to_numpy(float)
    z = p["elevacion_m"].to_numpy(float)
    area = p["area_m2"].to_numpy(float)

    chi = cumulative_trapezoid(
        (A0_VISUAL_M2 / area) ** THETA_VISUAL,
        x=x,
        initial=0.0,
    )

    fig, axs = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

    colores_terrazas = [
        "darkorange", "seagreen", "purple",
        "sienna", "darkcyan", "goldenrod",
    ]
    colores_kp = ["crimson", "black", "magenta", "darkgreen"]

    for ax, vista in zip(axs, ["distancia", "chi"]):
        horizontal = x / 1000 if vista == "distancia" else chi

        ax.plot(
            horizontal, z,
            color="steelblue", linewidth=1.4,
            label="Perfil fluvial",
        )

        for numero, (_, terraza) in enumerate(terrazas_rio.iterrows()):
            cota = float(terraza["Elevación (m)"])
            error = terraza["Error elevación (m)"]
            color = colores_terrazas[numero % len(colores_terrazas)]

            ax.axhline(
                cota, color=color, linestyle="--", linewidth=1.2,
                label=f"{terraza['terraza_id']}: {cota:.2f} m",
            )

            if pd.notna(error) and np.isfinite(error) and error > 0:
                ax.axhspan(
                    cota - error, cota + error,
                    color=color, alpha=0.10,
                )

        for numero, (metodo, grupo) in enumerate(
            kp_rio.groupby("metodo", sort=False)
        ):
            xkp = (
                grupo["distancia_m"].to_numpy(float) / 1000
                if vista == "distancia"
                else np.interp(grupo["distancia_m"], x, chi)
            )

            ax.scatter(
                xkp, grupo["elevacion_m"],
                color=colores_kp[numero % len(colores_kp)],
                marker="*" if seleccion else "o",
                s=180 if seleccion else 45,
                edgecolors="white", linewidths=0.5,
                label="Knickpoint seleccionado" if seleccion else str(metodo),
                zorder=5,
            )

            if seleccion:
                for posicion, (_, punto) in zip(xkp, grupo.iterrows()):
                    ax.annotate(
                        punto["id_kp"],
                        (posicion, punto["elevacion_m"]),
                        xytext=(5, 10),
                        textcoords="offset points",
                        fontsize=8,
                    )

        ax.set_xlabel(
            "Distancia desde el outlet (km)"
            if vista == "distancia" else r"$\chi$ (m)"
        )
        ax.set_title(
            "Perfil longitudinal"
            if vista == "distancia"
            else rf"Perfil $\chi$–elevación | $\theta={THETA_VISUAL:g}$"
        )
        ax.grid(alpha=0.2)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axs[0].set_ylabel("Elevación (m)")

    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="lower center",
        ncol=3,
        fontsize=8,
        frameon=False,
    )

    fig.suptitle(f"{rid} — {nombres[rid]} | {titulo}")
    margen_inferior = min(0.35, 0.05 * ((len(labels) + 2) // 3))
    fig.tight_layout(rect=(0, margen_inferior, 1, 0.94))

    plt.show()
    plt.close(fig)


for rid in map(str, RIOS_VER):
    if rid not in perfiles:
        raise ValueError(f"No existe el río {rid}.")

    ts = terrazas.loc[terrazas["ID"].eq(rid)]
    ks = knickpoints.loc[knickpoints["ID"].eq(rid)]

    graficar_terrazas_kp(rid, ts, ks)

    if MOSTRAR_TABLAS:
        print(f"\n{rid} — {nombres[rid]}")

        display(ts[[
            "terraza_id", "Elevación (m)", "Error elevación (m)",
        ]])

        display(
            ks[[
                "id_kp", "metodo", "distancia_m", "elevacion_m",
            ]].sort_values("distancia_m")
        )

## 4. Elegir knickpoint y terraza manualmente en Excel

Registramos una fila por río en la hoja `Hoja1` del archivo
`terrazas_kp_por_rio.xlsx`.

| Columna | Contenido |
|---|---|
| `id` | ID del perfil fluvial |
| `Cauce` | Nombre del río |
| `Terraza` | ID de la terraza, por ejemplo `JM2015_042` |
| `MIS` | Estadio asignado a la terraza |
| `Tipo` | Tipo de asignación cronológica |
| `Tasa Uplift` | Información original; pendiente de aclarar significado y unidades |
| `Observaciones` | Criterio de selección y dudas |
| `ID knickpoint` | Identificador exacto del knickpoint seleccionado |
| `Elevación (m)` | Elevación del knickpoint |
| `Este UTM (m)` | Coordenada Este del knickpoint |
| `Norte UTM (m)` | Coordenada Norte del knickpoint |

Las coordenadas de los knickpoints corresponden a EPSG:32718.

Cuando no exista una terraza seleccionada, dejar la celda vacía o escribir
`No tiene terraza cercana`. Cuando falte un knickpoint, dejar sus campos vacíos.

La cota de la terraza se obtiene de la planilla revisada mediante su ID
y su asociación al río. La columna `Elevación (m)` de este Excel corresponde
al knickpoint, no a la terraza.

La columna `Tasa Uplift` no se utiliza para calcular ni para dibujar la
elevación de las terrazas.

Guardar el Excel antes de ejecutar la siguiente celda.
No se requiere una segunda hoja.

In [ ]:
import hashlib

# ── RUTA DEL EXCEL DE SELECCIÓN MANUAL ────────────────────────────
RUTA_KP_TERRACE = r"C:\datos\Memoria\Codigos Stanford\terrazas_kp_por_rio.xlsx"

seleccion_excel = pd.read_excel(
    RUTA_KP_TERRACE,
    sheet_name="Hoja1",
).dropna(how="all")


def texto_excel(valor):
    return "" if pd.isna(valor) else str(valor).strip()


def normalizar_id_excel(valor):
    numero = float(valor)

    if not np.isfinite(numero) or not numero.is_integer():
        raise ValueError(f"ID de río inválido: {valor}")

    return str(int(numero))


requeridas = {
    "id", "Cauce", "Terraza", "ID knickpoint",
    "Elevación (m)", "Este UTM (m)", "Norte UTM (m)",
}

faltantes = requeridas - set(seleccion_excel.columns)

if faltantes:
    raise ValueError(f"Faltan columnas en Hoja1: {faltantes}")

seleccion_excel["ID"] = seleccion_excel["id"].map(normalizar_id_excel)

if seleccion_excel["ID"].duplicated().any():
    raise ValueError("Hoja1 debe contener una sola fila por río.")

filas_resumen = []
filas_terrazas = []
filas_kp = []

for _, fila in seleccion_excel.iterrows():
    rid = fila["ID"]

    if rid not in perfiles:
        raise ValueError(f"El río {rid} no está entre los perfiles cargados.")

    if normalizar_nombre(fila["Cauce"]) != normalizar_nombre(nombres[rid]):
        raise ValueError(f"Río {rid}: el nombre del Excel no coincide.")

    terraza_id = texto_excel(fila["Terraza"]).replace("-", "_")
    kp_id = texto_excel(fila["ID knickpoint"])
    pendientes = []

    # ── Terraza seleccionada ─────────────────────────────────────
    if not terraza_id or terraza_id.lower().startswith("no tiene terraza"):
        terraza_id = ""
        pendientes.append("Sin terraza seleccionada")

    else:
        ts = terrazas.loc[
            terrazas["ID"].eq(rid)
            & terrazas["terraza_id"].eq(terraza_id)
        ]

        if len(ts) != 1:
            raise ValueError(
                f"Río {rid}: {terraza_id} no tiene una asociación única "
                "con este río en la planilla revisada."
            )

        filas_terrazas.append(ts.iloc[0].to_dict())

    # ── Knickpoint seleccionado ──────────────────────────────────
    if not kp_id:
        pendientes.append("Sin knickpoint seleccionado")

    else:
        p = perfiles[rid]

        ks = knickpoints.loc[
            knickpoints["ID"].eq(rid)
            & knickpoints["id_kp"].eq(kp_id)
        ]

        if len(ks) == 1:
            kp = ks.iloc[0].to_dict()
            kp["origen_seleccion"] = "Catálogo exportado"

        else:
            # Formato de los IDs generados por la selección manual.
            coincidencia = re.fullmatch(
                r"R(\d+)_M_I(\d+)_([0-9a-f]{8})",
                kp_id,
            )

            if coincidencia is None:
                raise ValueError(
                    f"{kp_id}: no aparece en el catálogo y no tiene "
                    "el formato de un ID manual."
                )

            if int(coincidencia[1]) != int(rid):
                raise ValueError(f"{kp_id}: el ID pertenece a otro río.")

            j = int(coincidencia[2])

            if not 0 <= j < len(p):
                raise ValueError(f"{kp_id}: índice fuera del perfil.")

            datos_hash = np.asarray(
                p[[
                    "distancia_m", "elevacion_m", "area_m2",
                    "x_utm", "y_utm",
                ]].to_numpy(float),
                dtype="<f8",
                order="C",
            )

            huella = hashlib.sha256(datos_hash.tobytes()).hexdigest()[:8]

            if huella != coincidencia[3]:
                raise ValueError(
                    f"{kp_id}: la huella no coincide con el perfil cargado. "
                    "Revisa que estés utilizando la misma exportación."
                )

            kp = {
                **p.iloc[j].to_dict(),
                "ID": rid,
                "id_kp": kp_id,
                "indice": j,
                "metodo": "Manual (Excel)",
                "origen_seleccion": "Excel, validado contra el perfil",
            }

        # Comprobar elevación y coordenadas escritas en el Excel.
        equivalencias = {
            "Elevación (m)": "elevacion_m",
            "Este UTM (m)": "x_utm",
            "Norte UTM (m)": "y_utm",
        }

        for columna, campo in equivalencias.items():
            valor = float(fila[columna])

            if (
                not np.isfinite(valor)
                or not np.isclose(valor, kp[campo], atol=0.1, rtol=0)
            ):
                raise ValueError(
                    f"{kp_id}: '{columna}' no coincide con el perfil."
                )

        filas_kp.append(kp)

    filas_resumen.append({
        "ID": rid,
        "rio": nombres[rid],
        "terraza_id": terraza_id,
        "id_kp": kp_id,
        "estado": "; ".join(pendientes) or "Selección completa",
        "observaciones": texto_excel(fila.get("Observaciones")),
    })

tabla_seleccion = pd.DataFrame(filas_resumen)

terrazas_seleccionadas = pd.DataFrame(
    filas_terrazas,
    columns=terrazas.columns,
)

knickpoints_seleccionados = pd.DataFrame(
    filas_kp,
    columns=list(dict.fromkeys(
        list(knickpoints.columns) + ["origen_seleccion"]
    )),
)

display(tabla_seleccion)

print(
    f"Selección cargada: {len(tabla_seleccion)} ríos, "
    f"{len(terrazas_seleccionadas)} terrazas y "
    f"{len(knickpoints_seleccionados)} knickpoints."
)

## 5. Visualizar la selección manual del Excel

Mostramos la terraza y el knickpoint elegidos para cada río en
distancia–elevación y χ–elevación.

Los ríos con selección incompleta conservan el perfil y el elemento
disponible. No se asignan terrazas ni knickpoints automáticamente.

La tabla final reúne la selección, la cota de la terraza y las
coordenadas del knickpoint.

Para actualizar los gráficos, guardar el Excel y volver a ejecutar
las secciones 4 y 5.

In [ ]:
# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
RIOS_SELECCION_VER = tabla_seleccion["ID"].tolist()
# RIOS_SELECCION_VER = ["6"]

for rid in map(str, RIOS_SELECCION_VER):
    seleccion_rio = tabla_seleccion.loc[
        tabla_seleccion["ID"].eq(rid)
    ]

    if seleccion_rio.empty:
        raise ValueError(f"El río {rid} no tiene una fila en el Excel.")

    ts = terrazas_seleccionadas.loc[
        terrazas_seleccionadas["ID"].eq(rid)
    ]
    ks = knickpoints_seleccionados.loc[
        knickpoints_seleccionados["ID"].eq(rid)
    ]

    graficar_terrazas_kp(
        rid,
        ts,
        ks,
        titulo=seleccion_rio.iloc[0]["estado"],
        seleccion=True,
    )

# ── TABLA FINAL ──────────────────────────────────────────────────
resumen_seleccion = tabla_seleccion.merge(
    knickpoints_seleccionados[[
        "ID", "id_kp", "distancia_m",
        "elevacion_m", "x_utm", "y_utm",
    ]],
    on=["ID", "id_kp"],
    how="left",
    validate="one_to_one",
).merge(
    terrazas_seleccionadas[[
        "ID", "terraza_id", "Elevación (m)",
    ]].rename(columns={
        "Elevación (m)": "cota_terraza_m",
    }),
    on=["ID", "terraza_id"],
    how="left",
    validate="one_to_one",
)

display(
    resumen_seleccion.rename(columns={
        "ID": "ID río",
        "rio": "Río",
        "terraza_id": "Terraza",
        "cota_terraza_m": "Cota terraza (m)",
        "id_kp": "Knickpoint",
        "distancia_m": "Distancia KP (m)",
        "elevacion_m": "Elevación KP (m)",
        "x_utm": "Este KP (m)",
        "y_utm": "Norte KP (m)",
        "estado": "Estado",
        "observaciones": "Observaciones",
    }).round(2)
)